<a href="https://colab.research.google.com/github/vkantimahanti/healthcare-ml-portfolio/blob/main/Hyperparameters_gridsearchcv_breastcancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install mlflow shap -q

'''
mlflow - Installs MLflow, an open-source platform used to manage the machine learning lifecycle, including experiment tracking, model packaging, and deployment.
shap - Installs SHAP (SHapley Additive exPlanations), a library used for "model explainability." it helps you understand how different features in your data contribute to a model's specific predictions.
-q -  A flag that tells pip to be "quiet." It hides non-error messages (like progress bars and successful download notifications) to keep your terminal or notebook output clean.
'''

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 13.4 MB/s eta 0:00:00


'\nmlflow - Installs MLflow, an open-source platform used to manage the machine learning lifecycle, including experiment tracking, model packaging, and deployment.\nshap - Installs SHAP (SHapley Additive exPlanations), a library used for "model explainability." it helps you understand how different features in your data contribute to a model\'s specific predictions.\n-q -  A flag that tells pip to be "quiet." It hides non-error messages (like progress bars and successful download notifications) to keep your terminal or notebook output clean.\n'

In [1]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, roc_auc_score
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

GridSearchCV on your Thursday pipeline

In [2]:
# Reload data
data = load_breast_cancer()
X    = pd.DataFrame(data.data, columns=data.feature_names)
y    = pd.Series(data.target, name='cancer')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Base pipeline — same as Thursday
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler()),
    ('model',   RandomForestClassifier(random_state=42, class_weight='balanced'))
])

# Grid of values to try
# Note: prefix 'model__' tells GridSearch these params belong to the model step
param_grid = {
    'model__n_estimators' : [50, 100, 200],
    'model__max_depth'    : [3, 5, 7, None],
    'model__min_samples_split': [2, 5, 10]
}

# scoring='recall' — optimise for catching sick patients
# cv=5 — 5-fold cross validation on each combination
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='recall',       # healthcare priority
    n_jobs=-1,              # use all CPU cores — faster
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters : {grid_search.best_params_}")
print(f"Best CV Recall  : {grid_search.best_score_:.3f}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best parameters : {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Best CV Recall  : 0.968


Compare default vs tuned

In [3]:
from sklearn.metrics import (recall_score, precision_score,
                              roc_auc_score, f1_score)

# Default pipeline — Thursday's result
default_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler()),
    ('model',   RandomForestClassifier(
                    n_estimators=100, max_depth=5,
                    random_state=42, class_weight='balanced'))
])
default_pipeline.fit(X_train, y_train)
y_pred_default = default_pipeline.predict(X_test)

# Tuned pipeline — GridSearchCV best
best_pipeline  = grid_search.best_estimator_
y_pred_tuned   = best_pipeline.predict(X_test)
y_prob_tuned   = best_pipeline.predict_proba(X_test)[:, 1]

print("Default vs Tuned pipeline:")
print(f"{'Metric':12} {'Default':>10} {'Tuned':>10} {'Change':>10}")
print("-" * 45)
for metric_name, fn in [
    ('Recall',    recall_score),
    ('Precision', precision_score),
    ('F1',        f1_score)
]:
    default_score = fn(y_test, y_pred_default)
    tuned_score   = fn(y_test, y_pred_tuned)
    change        = tuned_score - default_score
    arrow         = "↑" if change > 0 else "↓" if change < 0 else "→"
    print(f"{metric_name:12} {default_score:>10.3f} {tuned_score:>10.3f} "
          f"{arrow} {abs(change):.3f}")

auc_tuned = roc_auc_score(y_test, y_prob_tuned)
print(f"{'ROC-AUC':12} {'0.994':>10} {auc_tuned:>10.3f}")

Default vs Tuned pipeline:
Metric          Default      Tuned     Change
---------------------------------------------
Recall            0.931      0.958 ↑ 0.028
Precision         0.957      0.958 ↑ 0.001
F1                0.944      0.958 ↑ 0.015
ROC-AUC           0.994      0.994


See what GridSearch tried

In [4]:
# See top 5 combinations GridSearch tested
results_df = pd.DataFrame(grid_search.cv_results_)
top5 = (results_df[['param_model__n_estimators',
                      'param_model__max_depth',
                      'param_model__min_samples_split',
                      'mean_test_score',
                      'std_test_score']]
        .sort_values('mean_test_score', ascending=False)
        .head(5)
        .round(3))

print("Top 5 parameter combinations:")
print(top5.to_string(index=False))
print(f"\nTotal combinations tested: {len(results_df)}")

Top 5 parameter combinations:
 param_model__n_estimators param_model__max_depth  param_model__min_samples_split  mean_test_score  std_test_score
                       200                   None                               2            0.968           0.013
                        50                      7                               5            0.965           0.011
                        50                   None                               5            0.965           0.011
                       100                      5                               2            0.965           0.016
                       200                      7                               2            0.965           0.016

Total combinations tested: 36


Save tuned pipeline + push

In [7]:
import joblib
import mlflow
import mlflow.sklearn

mlflow.set_experiment("breast-cancer-gridsearch")

with mlflow.start_run(run_name="random-forest-gridsearch"):
    best_params = grid_search.best_params_
    for param, value in best_params.items():
        mlflow.log_param(param.replace('model__', ''), value)
    mlflow.log_metric("best_cv_recall", round(grid_search.best_score_, 3))
    mlflow.log_metric("test_recall",
                      round(recall_score(y_test, y_pred_tuned), 3))
    mlflow.log_metric("test_roc_auc",
                      round(roc_auc_score(y_test, y_prob_tuned), 3))
    mlflow.sklearn.log_model(best_pipeline, "tuned-pipeline")

joblib.dump(best_pipeline, 'breast_cancer_tuned_pipeline.pkl')
print("Tuned pipeline saved ✓")

!git add .
!git commit -m "feat: gridsearchcv hyperparameter tuning on breast cancer pipeline"
!git push origin main
print("Pushed ✓ — GridSearchCV done!")

2026/03/15 19:38:16 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/15 19:38:16 INFO mlflow.store.db.utils: Updating database tables
2026/03/15 19:38:22 INFO mlflow.tracking.fluent: Experiment with name 'breast-cancer-gridsearch' does not exist. Creating a new experiment.
2026/03/15 19:38:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/15 19:38:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Tuned pipeline saved ✓
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
Pushed ✓ — GridSearchCV done!
